# LLM QLoRA / PEFT validation

Run this notebook in a fresh Google Colab CUDA runtime. It validates the hardware-specific 4-bit path rather than assuming that an arbitrary runtime supports it.

The validation covers runtime detection, backend installation, NF4 4-bit model loading, k-bit preparation, LoRA attachment, trainable-parameter fraction, a forward pass, and adapter export.

In [ ]:
import platform, sys, subprocess
print('OS:', platform.system())
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info()
        print('VRAM free GiB:', round(free / 1024**3, 2))
        print('VRAM total GiB:', round(total / 1024**3, 2))
except Exception as exc:
    print('Torch probe failed:', exc)


In [ ]:
import torch
assert torch.cuda.is_available(), 'A CUDA GPU runtime is required for this QLoRA validation notebook.'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'transformers', 'peft', 'accelerate', 'bitsandbytes'])
print('QLoRA backend dependencies installed')


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'sshleifer/tiny-gpt2'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
)
print('4-bit model loaded:', model.__class__.__name__)
print('Quantization config:', model.config.quantization_config)


In [ ]:
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print('Trainable params:', trainable)
print('Total params:', total)
print('Trainable fraction:', trainable / total)
assert trainable > 0
assert trainable < total * 0.1
model.print_trainable_parameters()
print('PEFT adapter validation passed')


In [ ]:
inputs = tokenizer('coding standards for machine learning', return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
print('Forward pass logits:', tuple(outputs.logits.shape))
assert outputs.logits.ndim == 3

output_dir = '/content/qlora_validation_adapter'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print('Adapter export:', output_dir)
print('QLoRA load + k-bit preparation + PEFT + forward + export validation passed')
